# Six-way factor benchmark — Haniffa COVID-19 CD8 T cells (SemanticSCVI Geneformer + GenePT vs LDVAE+ vs scHPF vs cNMF vs EXPIMAP)

Trains six factor models on `haniffa_cd8_clean.h5ad` (CD8 T cells subsetted from Haniffa 2021 COVID-19 PBMC atlas — `CD8.EM`, `CD8.TE`, `CD8.Naive`, `CD8.Activated_IL7R`, `CD8.Proliferating`; ≈4000 HVG, raw counts) and benchmarks against `lib1_immune.gmt` + `lib2_cd8.gmt`.

Same pipeline as `four_way_benchmark.ipynb`; only the data paths + cache dirs differ. Both SemanticSCVI variants share architecture but use different gene-embedding priors:
- `semantic_geom`   — Geneformer V2 (1152-d gc104M token embeddings)
- `semantic_genept` — GenePT text-embedding-3-large (3072-d NCBI+UniProt summaries, Zenodo 10833191)

The sixth model — `expimap_k10` (scArches EXPIMAP) — uses a **pathway mask** instead of a gene-embedding prior. The mask is built from 10 hand-picked CD8-relevant HALLMARK terms (symbol→Ensembl translated from `lib1_immune.gmt`), giving 10 latent gene programs to match `N_LATENT=10` across the other models.

All architecture / training knobs are explicit in **Cell 2 (parameters)**.

In [ ]:
# ============================================================
# Parameters — edit here. All training/architecture knobs in one place.
# ============================================================
import hashlib
import json
from pathlib import Path


def _find_nb_dir() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        for cand in (
            p / "notebooks" / "benchmark_helpers.py",
            p / "benchmark_helpers.py",
            p / "scvi-tools-neural-nmf" / "notebooks" / "benchmark_helpers.py",
        ):
            if cand.exists():
                return cand.parent.resolve()
    raise FileNotFoundError(
        f"Could not locate benchmark_helpers.py from cwd={Path.cwd()}. "
        "Launch jupyter under the scvi-tools-neural-nmf tree, or set NB_DIR manually."
    )


def _semantic_cache_slug(kwargs, max_epochs, warmup_epochs, n_epochs_kl_warmup, hvg_top_n, n=10):
    """Stable 10-char hash of every param that affects the trained SemanticSCVI model.
    Change any of these and the cache_dir flips -> fresh train. Same params -> cache hit."""
    blob = json.dumps(
        {
            "kwargs": dict(sorted(kwargs.items())),
            "max_epochs": max_epochs,
            "warmup_epochs": warmup_epochs,
            "n_epochs_kl_warmup": n_epochs_kl_warmup,
            "hvg_top_n": hvg_top_n,
        },
        default=str, sort_keys=True,
    )
    return hashlib.sha1(blob.encode()).hexdigest()[:n]


NB_DIR = _find_nb_dir()
print(f"NB_DIR = {NB_DIR}")

ADATA_PATH = NB_DIR / "haniffa_cd8_clean.h5ad"
# Geneformer prior.
SEMANTIC_CACHE_GENEFORMER = NB_DIR / "haniffa_cd8_clean_geneformer.pt"
# GenePT prior — raw pickle (downloaded on first call) + per-adata aligned tensor cache.
SEMANTIC_CACHE_GENEPT     = NB_DIR / "haniffa_cd8_clean_genept_3large.pt"
GENEPT_PICKLE_PATH        = NB_DIR / "GenePT_gene_protein_embedding_model_3_text.pickle"

PATHWAY_INDEX = NB_DIR / "pathway_index.pkl"
LIB1_GMT = NB_DIR / "lib1_immune.gmt"
LIB2_GMT = NB_DIR / "lib2_cd8.gmt"
OUT_DIR = NB_DIR / "benchmark_results" / "haniffa_cd8_four_way"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preprocessing ----
HVG_TOP_N = 2500       # None = use all genes; int = take top-N HVGs (sc.pp.highly_variable_genes)
HVG_FLAVOR = "seurat_v3"  # works on raw counts; switch to "seurat" / "cell_ranger" if you log-normalized

# Optional subsample (mirrors zheng_cd8 notebook). None = use all CD8 cells.
SUBSAMPLE_N = 40000

# ---- Cache root ----
MODEL_CACHE_DIR = NB_DIR / ".model_cache_haniffa_cd8"

# ---- Shared model size ----
N_LATENT = 10

# ---- Batch effect (scvi-based models) ----
# scHPF / cNMF have no native batch support and ignore this knob.
BATCH_KEY = "Site"

# ---- SemanticSCVI (Geneformer prior) ----
SEMANTIC_GENEFORMER_MAX_EPOCHS = 200
SEMANTIC_GENEFORMER_WARMUP_EPOCHS = 20
SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP = 100
SEMANTIC_GENEFORMER_KWARGS = dict(
    loss_mode="geometric",
    coherence_weight=2000.0,
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- SemanticSCVI (GenePT prior) ----
SEMANTIC_GENEPT_MAX_EPOCHS = 200
SEMANTIC_GENEPT_WARMUP_EPOCHS = 20
SEMANTIC_GENEPT_N_EPOCHS_KL_WARMUP = 100
SEMANTIC_GENEPT_KWARGS = dict(
    loss_mode="geometric",
    coherence_weight=2000.0,
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# ---- Param-aware cache_dirs ----
# Fold BATCH_KEY into the hash so flipping the batch key auto-invalidates the cache.
SEMANTIC_GENEFORMER_CACHE_DIR = MODEL_CACHE_DIR / "semantic_scvi" / _semantic_cache_slug(
    {**SEMANTIC_GENEFORMER_KWARGS, "batch_key": BATCH_KEY},
    SEMANTIC_GENEFORMER_MAX_EPOCHS,
    SEMANTIC_GENEFORMER_WARMUP_EPOCHS,
    SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP,
    HVG_TOP_N,
)
SEMANTIC_GENEPT_CACHE_DIR = MODEL_CACHE_DIR / "semantic_scvi_genept" / _semantic_cache_slug(
    {**SEMANTIC_GENEPT_KWARGS, "batch_key": BATCH_KEY},
    SEMANTIC_GENEPT_MAX_EPOCHS,
    SEMANTIC_GENEPT_WARMUP_EPOCHS,
    SEMANTIC_GENEPT_N_EPOCHS_KL_WARMUP,
    HVG_TOP_N,
)
print(f"semantic_geom   cache_dir: {SEMANTIC_GENEFORMER_CACHE_DIR}")
print(f"semantic_genept cache_dir: {SEMANTIC_GENEPT_CACHE_DIR}")

# ---- Force-retrain switches ----
FORCE_TRAIN_SEMANTIC_GENEFORMER = False
FORCE_TRAIN_SEMANTIC_GENEPT     = False
FORCE_TRAIN_LDVAE = False
FORCE_TRAIN_SCHPF = False
FORCE_TRAIN_NMF = False
FORCE_TRAIN_EXPIMAP = False

# ---- LDVAE+ training ----
LDVAE_MAX_EPOCHS = 250
LDVAE_KWARGS = dict(
    n_hidden=128,
    n_latent=N_LATENT,
    n_layers=1,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)
# Bake BATCH_KEY into LDVAE cache dir (no param hash on this wrapper).
LDVAE_CACHE_DIR = MODEL_CACHE_DIR / f"ldvae_nn_batch_{BATCH_KEY or 'none'}"

# ---- scHPF / cNMF ----
N_FACTORS = N_LATENT
NMF_INIT = "nndsvd"
NMF_MAX_ITER = 500
NMF_RANDOM_STATE = 42

# ---- EXPIMAP (scArches) — pathway-mask conditional VAE ----
# 10 CD8-relevant HALLMARK terms; mask built by symbol->Ensembl translation
# from lib1_immune.gmt. Latent dim = number of surviving terms (target 10).
EXPIMAP_MASK_TERMS = [
    "HALLMARK_INTERFERON_GAMMA_RESPONSE",
    "HALLMARK_IL2_STAT5_SIGNALING",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
    "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_OXIDATIVE_PHOSPHORYLATION",
    "HALLMARK_GLYCOLYSIS",
    "HALLMARK_MTORC1_SIGNALING",
    "HALLMARK_HYPOXIA",
    "HALLMARK_E2F_TARGETS",
    "HALLMARK_G2M_CHECKPOINT",
]
EXPIMAP_SOURCE_GMT = LIB1_GMT
EXPIMAP_MASK_GMT = NB_DIR / f"{ADATA_PATH.stem}_expimap_hallmark.gmt"
EXPIMAP_N_EPOCHS = 400
EXPIMAP_ALPHA = 0.7
EXPIMAP_ALPHA_KL = 0.5
EXPIMAP_ALPHA_EPOCH_ANNEAL = 130
EXPIMAP_KWARGS = dict(
    condition_key=BATCH_KEY,       # NB recon path requires batch one-hot (scarches/models/expimap/expimap.py:194)
    recon_loss="nb",               # requires raw counts in adata.X (true after HVG block)
    hidden_layer_sizes=(256, 256),
)
EXPIMAP_CACHE_DIR = MODEL_CACHE_DIR / "expimap_k10" / _semantic_cache_slug(
    {
        **EXPIMAP_KWARGS,
        "terms": tuple(EXPIMAP_MASK_TERMS),
        "alpha": EXPIMAP_ALPHA,
        "alpha_kl": EXPIMAP_ALPHA_KL,
        "alpha_epoch_anneal": EXPIMAP_ALPHA_EPOCH_ANNEAL,
    },
    EXPIMAP_N_EPOCHS, 0, EXPIMAP_ALPHA_EPOCH_ANNEAL, HVG_TOP_N,
)
print(f"expimap_k10     cache_dir: {EXPIMAP_CACHE_DIR}")

# ---- Benchmark / report ----
MODEL_NAMES = ["semantic_geom", "semantic_genept", "ldvae_nn", "schpf_k10", "nmf_k10", "expimap_k10"]
PER_PROJECTION_N_TOP = 30
CLUSTER_N_TOP = 500     # pool of top-loaded genes fed into UMAP+Leiden and hierarchical clustering
CLUSTER_TOP_PER_CLUSTER = 30  # int = after Leiden, keep only top-N genes per cluster (by max_loading)
GENE_MAPPING = ("feature_id", "feature_name")  # Ensembl → symbol via adata.var columns
ENABLE_LLM_GRADING = True  # False = skip Claude Opus + Gemini Pro LLM scoring (also skips API-key prompt)


In [ ]:
import sys
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import scanpy as sc
import torch
from sklearn.decomposition import NMF
from scipy.sparse import coo_matrix, issparse
import numpy as np
import schpf

# scArches 0.6.1 imports `from anndata import read`, which newer anndata dropped.
# Shim it before importing scarches — read_h5ad is the modern replacement.
import anndata as _ad
if not hasattr(_ad, "read"):
    _ad.read = _ad.read_h5ad

# Lazy install: scArches isn't a default project dep; only needed for the expimap_k10 baseline.
try:
    import scarches as sca
except ImportError:
    %pip install --quiet scarches==0.6.1
    import scarches as sca
print(f"scarches {sca.__version__}")

from benchmark_helpers import (
    NMFWrapper,
    _ExpimapAdapter,
    _ScviAdapter,
    build_expimap_mask_gmt,
    build_report,
    get_or_build_geneformer_map,
    get_or_build_genept_map,
    train_or_load_expimap,
    train_or_load_nonneg_ldvae,
    train_or_load_pickle,
    train_or_load_semantic_scvi,
)
from benchmarking import SemanticBenchmark
from train_schpf import train_nmf_model, train_schpf_model

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"NB_DIR = {NB_DIR}")

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print("adata (raw):", adata.shape)
print("var cols:", list(adata.var.columns))
print("obs cols:", list(adata.obs.columns))
print("cell_type counts:\n", adata.obs["cell_type"].value_counts())

# Build/load the FULL-gene semantic maps for both priors first, then HVG-subset all of them together.
semantic_map_geneformer = get_or_build_geneformer_map(adata, SEMANTIC_CACHE_GENEFORMER)
print("semantic_map (geneformer, raw):", tuple(semantic_map_geneformer.shape))

semantic_map_genept = get_or_build_genept_map(adata, SEMANTIC_CACHE_GENEPT, GENEPT_PICKLE_PATH)
print("semantic_map (genept, raw):", tuple(semantic_map_genept.shape))

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(
        adata, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False
    )
    keep = adata.var["highly_variable"].values
    adata = adata[:, keep].copy()
    keep_t = torch.as_tensor(keep)
    semantic_map_geneformer = semantic_map_geneformer[keep_t]
    semantic_map_genept     = semantic_map_genept[keep_t]
    print(f"After HVG filter (top {HVG_TOP_N}, flavor={HVG_FLAVOR}):")
    print("  adata:", adata.shape)
    print("  semantic_map (geneformer):", tuple(semantic_map_geneformer.shape))
    print("  semantic_map (genept):    ", tuple(semantic_map_genept.shape))
else:
    print("HVG filter skipped (HVG_TOP_N=None)")

In [ ]:
if SUBSAMPLE_N is not None and adata.n_obs > SUBSAMPLE_N:
    sc.pp.subsample(adata, n_obs=SUBSAMPLE_N, random_state=42)
    print(f"subsampled to {adata.n_obs} cells")
else:
    print(f"no subsampling (n_obs={adata.n_obs}, SUBSAMPLE_N={SUBSAMPLE_N})")

In [ ]:
# Reload helpers so the new batch_key parameter is picked up without a kernel restart.
import importlib, benchmark_helpers
importlib.reload(benchmark_helpers)
from benchmark_helpers import (
    train_or_load_semantic_scvi,
    train_or_load_nonneg_ldvae,
    train_or_load_expimap,
    train_or_load_pickle,
    build_expimap_mask_gmt,
    NMFWrapper,
)

# Each scvi model writes a UUID into adata.uns, so each needs its own copy.
adata_sem_geneformer = adata.copy()
semantic_geneformer_model = train_or_load_semantic_scvi(
    adata_sem_geneformer,
    semantic_map_geneformer,
    cache_dir=SEMANTIC_GENEFORMER_CACHE_DIR,
    force_train=FORCE_TRAIN_SEMANTIC_GENEFORMER,
    max_epochs=SEMANTIC_GENEFORMER_MAX_EPOCHS,
    warmup_epochs=SEMANTIC_GENEFORMER_WARMUP_EPOCHS,
    n_epochs_kl_warmup=SEMANTIC_GENEFORMER_N_EPOCHS_KL_WARMUP,
    batch_key=BATCH_KEY,
    **SEMANTIC_GENEFORMER_KWARGS,
)

adata_sem_genept = adata.copy()
semantic_genept_model = train_or_load_semantic_scvi(
    adata_sem_genept,
    semantic_map_genept,
    cache_dir=SEMANTIC_GENEPT_CACHE_DIR,
    force_train=FORCE_TRAIN_SEMANTIC_GENEPT,
    max_epochs=SEMANTIC_GENEPT_MAX_EPOCHS,
    warmup_epochs=SEMANTIC_GENEPT_WARMUP_EPOCHS,
    n_epochs_kl_warmup=SEMANTIC_GENEPT_N_EPOCHS_KL_WARMUP,
    batch_key=BATCH_KEY,
    **SEMANTIC_GENEPT_KWARGS,
)

adata_ldvae = adata.copy()
ldvae_model = train_or_load_nonneg_ldvae(
    adata_ldvae,
    cache_dir=LDVAE_CACHE_DIR,
    force_train=FORCE_TRAIN_LDVAE,
    max_epochs=LDVAE_MAX_EPOCHS,
    batch_key=BATCH_KEY,
    **LDVAE_KWARGS,
)

# scHPF / cNMF have no native batch-correction; BATCH_KEY is not forwarded.
schpf_model = train_or_load_pickle(
    "scHPF",
    lambda: train_schpf_model(adata, n_factors=N_FACTORS),
    cache_path=MODEL_CACHE_DIR / "schpf_k10.pkl",
    force_train=FORCE_TRAIN_SCHPF,
)

def _train_nmf():
    print(f"Training cNMF (sklearn NMF, k={N_FACTORS}, init={NMF_INIT})...")
    X_nmf = adata.X
    if issparse(X_nmf):
        if (X_nmf.data < 0).any():
            X_nmf.data = np.maximum(X_nmf.data, 0)
    else:
        X_nmf = np.maximum(X_nmf, 0)
    _nmf = NMF(
        n_components=N_FACTORS,
        init=NMF_INIT,
        random_state=NMF_RANDOM_STATE,
        max_iter=NMF_MAX_ITER,
    )
    W = _nmf.fit_transform(X_nmf)
    H = _nmf.components_
    return NMFWrapper(model=_nmf, W=W, H=H, feature_names=adata.var_names)

nmf_model = train_or_load_pickle(
    "cNMF/NMF",
    _train_nmf,
    cache_path=MODEL_CACHE_DIR / "nmf_k10.pkl",
    force_train=FORCE_TRAIN_NMF,
)

# ---- EXPIMAP (pathway-mask conditional VAE) ----
# condition_key = BATCH_KEY (set in EXPIMAP_KWARGS above).
# Build the symbol->Ensembl-translated mask GMT once per adata (cached on disk).
if not EXPIMAP_MASK_GMT.exists():
    build_expimap_mask_gmt(
        adata, EXPIMAP_SOURCE_GMT, EXPIMAP_MASK_TERMS, EXPIMAP_MASK_GMT,
    )
adata_expimap = adata.copy()
expimap_model = train_or_load_expimap(
    adata_expimap,
    gmt_path=EXPIMAP_MASK_GMT,
    cache_dir=EXPIMAP_CACHE_DIR,
    force_train=FORCE_TRAIN_EXPIMAP,
    n_epochs=EXPIMAP_N_EPOCHS,
    alpha_kl=EXPIMAP_ALPHA_KL,
    alpha=EXPIMAP_ALPHA,
    alpha_epoch_anneal=EXPIMAP_ALPHA_EPOCH_ANNEAL,
    **EXPIMAP_KWARGS,
)
print(f"expimap_k10 active terms: {len(expimap_model.nonzero_terms())} / {len(adata_expimap.uns['terms'])}")


In [ ]:
# ============================================================
# Training-loss curves — train/val for each SemanticSCVI variant and LDVAE+.
# scHPF / cNMF have no per-epoch history, so they're skipped.
# EXPIMAP: attempted with semantic=False; plot_training_curves silently skips
# if scArches' history dict doesn't match scvi's {metric: DataFrame} schema.
# ============================================================
import importlib, benchmark_helpers
importlib.reload(benchmark_helpers)
from benchmark_helpers import plot_training_curves

plot_training_curves(
    semantic_geneformer_model, "SemanticSCVI (Geneformer)",
    OUT_DIR / "training_curves_semantic.png",
    semantic=True,
)
plot_training_curves(
    semantic_genept_model, "SemanticSCVI (GenePT)",
    OUT_DIR / "training_curves_semantic_genept.png",
    semantic=True,
)
plot_training_curves(
    ldvae_model, "LDVAE+",
    OUT_DIR / "training_curves_ldvae.png",
    semantic=False,
)
plot_training_curves(
    expimap_model, "EXPIMAP",
    OUT_DIR / "training_curves_expimap.png",
    semantic=False,
)
print("scHPF / cNMF: no per-epoch training history (skipping).")

In [ ]:
models = {
    MODEL_NAMES[0]: _ScviAdapter(semantic_geneformer_model, adata_sem_geneformer),
    MODEL_NAMES[1]: _ScviAdapter(semantic_genept_model,     adata_sem_genept),
    MODEL_NAMES[2]: _ScviAdapter(ldvae_model, adata_ldvae),
    MODEL_NAMES[3]: schpf_model,
    MODEL_NAMES[4]: nmf_model,
    MODEL_NAMES[5]: _ExpimapAdapter(expimap_model, adata_expimap),
}
for name, m in models.items():
    z = m.get_latent_representation()
    print(f"{name}: latent shape = {z.shape}")

In [ ]:
# ============================================================
# API keys for LLM grading
# ------------------------------------------------------------
# Claude judge: uses the `claude -p` CLI (no API key — runs against your
#   Claude Code subscription, no extra charges).
# Gemini judge: needs GEMINI_API_KEY (get a free one at
#   https://aistudio.google.com/apikey).
# First run prompts (masked) and persists to .claude/api_keys.env
# (gitignored, chmod 600). Subsequent runs are silent.
# ============================================================
if ENABLE_LLM_GRADING:
    import os, getpass, shutil
    from pathlib import Path

    KEYS_FILE = Path(NB_DIR).parent / ".claude" / "api_keys.env"
    KEYS_FILE.parent.mkdir(parents=True, exist_ok=True)

    if KEYS_FILE.exists():
        for line in KEYS_FILE.read_text().splitlines():
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())

    if not os.environ.get("GEMINI_API_KEY") and not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ").strip()
        with KEYS_FILE.open("a") as fh:
            fh.write(f"GEMINI_API_KEY={os.environ['GEMINI_API_KEY']}\n")
        KEYS_FILE.chmod(0o600)

    # Lazy install of the Gemini SDK (one-time)
    try:
        from google import genai  # noqa: F401
    except ImportError:
        %pip install --quiet "google-genai>=1.51.0"
        from google import genai  # noqa: F401

    if shutil.which("claude") is None:
        print("WARNING: `claude` CLI not found on PATH — Claude judge will fail. "
              "Install Claude Code or set judges=... on SemanticBenchmark.")
    print("Gemini key loaded; Claude CLI ready.")
else:
    print("ENABLE_LLM_GRADING=False — skipping API-key setup.")


In [ ]:
bench = SemanticBenchmark(
    models,
    adata,
    pathway_index=str(PATHWAY_INDEX),
    gene_mapping=GENE_MAPPING,
    out_dir=str(OUT_DIR),
    cell_type_context=(
        "human tissue CD8 T cells (Haniffa COVID-19 PBMC atlas subset): mix of "
        "CD8.TE (terminal effector), CD8.EM (effector memory), CD8.Naive, "
        "CD8.Proliferating. Expect cytotoxic effector, exhaustion, IFN response, "
        "memory/naive identity, proliferation/cell-cycle programs."
    ),
)
# semantic_alignment is computed against a single reference embedding. We pass the
# Geneformer prior here (matches the original notebook) so the score is identical
# for ldvae/schpf/nmf/semantic_geom; semantic_genept is scored against a different
# prior than its own — interpret accordingly.
bench.run_figures(
    semantic_map=semantic_map_geneformer,
    lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
    lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
    per_projection_n_top=PER_PROJECTION_N_TOP,
    cluster_n_top=CLUSTER_N_TOP,
    cluster_top_per_cluster=CLUSTER_TOP_PER_CLUSTER,
)


In [ ]:
if ENABLE_LLM_GRADING:
    bench.run_grading(
        lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
        lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
        per_projection_n_top=PER_PROJECTION_N_TOP,
        cluster_n_top=CLUSTER_N_TOP,
    )
else:
    print("LLM grading skipped (ENABLE_LLM_GRADING=False)")

In [ ]:
# ============================================================
# Max-program-per-factor enrichment summary.
# Per (model, library): ER + HG p per factor per program -> BH q (per model×lib)
# -> keep each factor's MAX-ER program. Two scalars (both over #factors):
#   pct_sig        = % factors whose best program is q-significant (q<0.05)
#   pct_sig_strong = % factors significant AND max_ER > ER_THRESH
# Then histogram of the max-ER vector, faceted library (rows) × method (cols).
# Reuses bench internals (CPU-only HG tests; no retraining).
#
# ER = (overlap/set_size) / (factor_size/G); theoretical max = G/factor_size.
# diag columns (overlap/set_size/factor_size/G) are kept so big-ER tails can be
# traced to tiny gene sets fully captured (high fold, few genes).
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests

from benchmarking import _HG_LIB_ORDER, _HG_LIB_TITLES

ER_THRESH = 2.0
Q_THRESH = 0.05

per_proj = bench._collect_top_genes_per_factor(PER_PROJECTION_N_TOP)
libs = bench._build_hg_libraries(str(LIB1_GMT), str(LIB2_GMT))
universe = bench._universe_symbols()
print(f"universe |G| = {len(universe)} symbols; per_projection_n_top = {PER_PROJECTION_N_TOP}")

max_rows = []
for label, library in libs.items():
    for model_name, factor_genes in per_proj.items():
        # Flatten all factor×program p-values for this (model, library) for BH.
        flat, ranges = [], []
        for factor, genes in factor_genes.items():
            recs = bench._hg_enrichment_rows(genes, library, universe)
            start = len(flat)
            flat.extend(r["pvalue"] for r in recs)
            ranges.append((factor, recs, start, len(flat)))
        if not flat:
            continue
        _, q_all, _, _ = multipletests(flat, method="fdr_bh")
        for factor, recs, start, end in ranges:
            if not recs:
                continue
            qs = q_all[start:end]
            best_i = max(range(len(recs)), key=lambda i: recs[i]["ER"])
            best = recs[best_i]
            q = float(qs[best_i])
            max_rows.append({
                "Model": model_name,
                "Library": label,
                "Factor": factor,
                "gene_set": best["gene_set"],
                "max_ER": best["ER"],
                "overlap": best["overlap"],
                "set_size": best["set_size"],
                "factor_size": best["factor_size"],
                "G": best["G"],
                "qvalue": q,
                "significant": bool(q < Q_THRESH),
            })

maxdf = pd.DataFrame(max_rows)

# --- Sanity checks: confirm we're on the loadings (factor_size ~ n_top) ---
print("\nfactor_size (|top genes ∩ universe|) describe:")
print(maxdf["factor_size"].describe().to_string())
dropped = maxdf[maxdf["factor_size"] < PER_PROJECTION_N_TOP]
if len(dropped):
    print(f"!! {len(dropped)} rows with factor_size < {PER_PROJECTION_N_TOP} "
          f"(duplicate symbols or genes off universe -> raises ER ceiling G/factor_size)")
print("\nTop 12 by max_ER (high fold = small set fully captured?):")
print(maxdf.sort_values("max_ER", ascending=False)
      .head(12)[["Model", "Library", "Factor", "gene_set",
                 "overlap", "set_size", "factor_size", "max_ER", "qvalue"]]
      .to_string(index=False))

# --- Summary table (per model × library; both percentages over #factors) ---
def _summ(g):
    n = len(g)
    return pd.Series({
        "n_factors": n,
        "pct_sig": 100.0 * g["significant"].mean(),
        "pct_sig_strong": 100.0 * ((g["significant"]) & (g["max_ER"] > ER_THRESH)).mean(),
    })

summary = (
    maxdf.groupby(["Library", "Model"]).apply(_summ, include_groups=False)
    .reset_index()
)
summary["Library"] = pd.Categorical(summary["Library"], categories=list(_HG_LIB_ORDER), ordered=True)
summary["Model"] = pd.Categorical(summary["Model"], categories=MODEL_NAMES, ordered=True)
summary = summary.sort_values(["Library", "Model"]).reset_index(drop=True)
print("\n" + summary.to_string(index=False))

# --- Histogram grid: rows = library, cols = method ---
color_map = dict(zip(MODEL_NAMES, sns.color_palette("tab10", len(MODEL_NAMES))))
libs_present = [l for l in _HG_LIB_ORDER if l in maxdf["Library"].unique()]
n_r, n_c = len(libs_present), len(MODEL_NAMES)
fig, axes = plt.subplots(n_r, n_c, figsize=(2.6 * n_c, 2.6 * n_r),
                         squeeze=False, sharex="row")
for ri, lib in enumerate(libs_present):
    er_max = maxdf.loc[maxdf["Library"] == lib, "max_ER"].max()
    bins = np.linspace(0, float(er_max) if er_max == er_max else 1.0, 16)
    for ci, model_name in enumerate(MODEL_NAMES):
        ax = axes[ri][ci]
        v = maxdf[(maxdf["Library"] == lib) & (maxdf["Model"] == model_name)]
        ax.hist(v["max_ER"], bins=bins, color=color_map[model_name],
                edgecolor="white", alpha=0.9)
        ax.axvline(ER_THRESH, ls="--", lw=1, color="#444")
        row = summary[(summary["Library"] == lib) & (summary["Model"] == model_name)]
        if len(row):
            r = row.iloc[0]
            ax.set_title(f"{model_name}\nsig {r.pct_sig:.0f}% / strong {r.pct_sig_strong:.0f}%",
                         fontsize=8)
        else:
            ax.set_title(model_name, fontsize=8)
        if ci == 0:
            ax.set_ylabel(f"{_HG_LIB_TITLES.get(lib, lib)}\ncount", fontsize=8)
        if ri == n_r - 1:
            ax.set_xlabel("max fold enrichment", fontsize=8)
        ax.tick_params(labelsize=7)
fig.suptitle("Per-factor best-program enrichment — max ER per factor "
             f"(top-{PER_PROJECTION_N_TOP} genes, ER>{ER_THRESH} = strong)", y=1.01)
fig.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Smoothed (KDE) overlay of the max-ER vectors — one panel per library,
# all methods on top of each other for direct comparison.
# Uses maxdf / color_map / libs_present from the previous cell.
# ============================================================
fig, axes = plt.subplots(1, len(libs_present), figsize=(5.0 * len(libs_present), 4.0),
                         squeeze=False)
for ci, lib in enumerate(libs_present):
    ax = axes[0][ci]
    sub = maxdf[maxdf["Library"] == lib]
    for model_name in MODEL_NAMES:
        v = sub.loc[sub["Model"] == model_name, "max_ER"].dropna()
        if v.nunique() < 2:   # kde needs spread
            continue
        sns.kdeplot(v, ax=ax, color=color_map[model_name], label=model_name,
                    fill=False, lw=1.8, clip=(0, None))
    ax.axvline(ER_THRESH, ls="--", lw=1, color="#444")
    ax.set_title(_HG_LIB_TITLES.get(lib, lib))
    ax.set_xlabel("max fold enrichment")
    ax.set_ylabel("density")
    if ci == len(libs_present) - 1:
        ax.legend(fontsize=7, title="method")
fig.suptitle(f"Per-factor best-program enrichment — smoothed max-ER (top-{PER_PROJECTION_N_TOP} genes)",
             y=1.02)
fig.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Same max-program analysis, but on GENE CLUSTERS instead of factors.
# Two clustering methods (umap_leiden, hierarchical) — same ones graded above;
# cluster gene lists pulled from bench.cluster_cache (no recompute).
# Per (method, model, library): ER+p per cluster per program -> BH q
# (per method×model×lib) -> keep each cluster's MAX-ER program.
#   pct_sig        = % clusters whose best program is q-significant
#   pct_sig_strong = % clusters significant AND max_ER > ER_THRESH (both over #clusters)
# Histogram grid (library × model) + KDE overlay, one figure per method.
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests

from benchmarking import _HG_LIB_ORDER, _HG_LIB_TITLES

ER_THRESH = 2.0
Q_THRESH = 0.05
libs = bench._build_hg_libraries(str(LIB1_GMT), str(LIB2_GMT))
universe = bench._universe_symbols()
color_map = dict(zip(MODEL_NAMES, sns.color_palette("tab10", len(MODEL_NAMES))))

# --- 1. Collect cluster gene lists per (method, model) — same as run_grading ---
per_method = {}  # {method: {model: {cluster_id: [symbols]}}}
for method_name in ("umap_leiden", "hierarchical"):
    per_method[method_name] = {}
    for model_name, model in bench.models.items():
        if method_name == "umap_leiden":
            adata_genes = bench._get_gene_clusters(model_name, model, n_top=CLUSTER_N_TOP, resolution=1.0)
            cluster_df = bench._leiden_to_cluster_df(adata_genes)
        else:
            cluster_df = bench._get_hierarchical_clusters(model_name, model, n_top=CLUSTER_N_TOP, max_k=20)
        if cluster_df is None or cluster_df.empty:
            continue
        ctg = bench._top_symbols_per_cluster(
            cluster_df, cluster_col="Cluster",
            n_top_per_cluster=CLUSTER_TOP_PER_CLUSTER, min_cluster_size=5,
        )
        if ctg:
            per_method[method_name][model_name] = ctg

# --- 2. Enrichment -> max program per cluster (BH per method×model×library) ---
rows = []
for method_name, models_clusters in per_method.items():
    for label, library in libs.items():
        for model_name, ctg in models_clusters.items():
            flat, ranges = [], []
            for cid, genes in ctg.items():
                recs = bench._hg_enrichment_rows(genes, library, universe)
                start = len(flat)
                flat.extend(r["pvalue"] for r in recs)
                ranges.append((cid, recs, start, len(flat)))
            if not flat:
                continue
            _, q_all, _, _ = multipletests(flat, method="fdr_bh")
            for cid, recs, start, end in ranges:
                if not recs:
                    continue
                qs = q_all[start:end]
                best_i = max(range(len(recs)), key=lambda i: recs[i]["ER"])
                best = recs[best_i]
                q = float(qs[best_i])
                rows.append({
                    "Method": method_name, "Model": model_name, "Library": label,
                    "Cluster": cid, "gene_set": best["gene_set"], "max_ER": best["ER"],
                    "overlap": best["overlap"], "set_size": best["set_size"],
                    "factor_size": best["factor_size"], "G": best["G"],
                    "qvalue": q, "significant": bool(q < Q_THRESH),
                })

cluster_maxdf = pd.DataFrame(rows)

# --- 3. Summary table per (Method, Library, Model) ---
def _summ(g):
    return pd.Series({
        "n_clusters": len(g),
        "pct_sig": 100.0 * g["significant"].mean(),
        "pct_sig_strong": 100.0 * ((g["significant"]) & (g["max_ER"] > ER_THRESH)).mean(),
    })

cluster_summary = (
    cluster_maxdf.groupby(["Method", "Library", "Model"]).apply(_summ, include_groups=False)
    .reset_index()
)
cluster_summary["Library"] = pd.Categorical(cluster_summary["Library"], categories=list(_HG_LIB_ORDER), ordered=True)
cluster_summary["Model"] = pd.Categorical(cluster_summary["Model"], categories=MODEL_NAMES, ordered=True)
cluster_summary = cluster_summary.sort_values(["Method", "Library", "Model"]).reset_index(drop=True)
print(cluster_summary.to_string(index=False))


# --- 4. Plots — one histogram grid + one KDE overlay per method ---
def _hist_grid(df, summ, title):
    libs_present = [l for l in _HG_LIB_ORDER if l in df["Library"].unique()]
    n_r, n_c = len(libs_present), len(MODEL_NAMES)
    fig, axes = plt.subplots(n_r, n_c, figsize=(2.6 * n_c, 2.6 * n_r), squeeze=False, sharex="row")
    for ri, lib in enumerate(libs_present):
        er_max = df.loc[df["Library"] == lib, "max_ER"].max()
        bins = np.linspace(0, float(er_max) if er_max == er_max else 1.0, 16)
        for ci, model_name in enumerate(MODEL_NAMES):
            ax = axes[ri][ci]
            v = df[(df["Library"] == lib) & (df["Model"] == model_name)]
            ax.hist(v["max_ER"], bins=bins, color=color_map[model_name], edgecolor="white", alpha=0.9)
            ax.axvline(ER_THRESH, ls="--", lw=1, color="#444")
            r = summ[(summ["Library"] == lib) & (summ["Model"] == model_name)]
            if len(r):
                r = r.iloc[0]
                ax.set_title(f"{model_name}\nsig {r.pct_sig:.0f}% / strong {r.pct_sig_strong:.0f}% (n={int(r.n_clusters)})", fontsize=7)
            else:
                ax.set_title(model_name, fontsize=8)
            if ci == 0:
                ax.set_ylabel(f"{_HG_LIB_TITLES.get(lib, lib)}\ncount", fontsize=8)
            if ri == n_r - 1:
                ax.set_xlabel("max fold enrichment", fontsize=8)
            ax.tick_params(labelsize=7)
    fig.suptitle(title, y=1.01)
    fig.tight_layout(); plt.show()

def _kde_overlay(df, title):
    libs_present = [l for l in _HG_LIB_ORDER if l in df["Library"].unique()]
    fig, axes = plt.subplots(1, len(libs_present), figsize=(5.0 * len(libs_present), 4.0), squeeze=False)
    for ci, lib in enumerate(libs_present):
        ax = axes[0][ci]
        sub = df[df["Library"] == lib]
        for model_name in MODEL_NAMES:
            v = sub.loc[sub["Model"] == model_name, "max_ER"].dropna()
            if v.nunique() < 2:
                continue
            sns.kdeplot(v, ax=ax, color=color_map[model_name], label=model_name, lw=1.8, clip=(0, None))
        ax.axvline(ER_THRESH, ls="--", lw=1, color="#444")
        ax.set_title(_HG_LIB_TITLES.get(lib, lib)); ax.set_xlabel("max fold enrichment"); ax.set_ylabel("density")
        if ci == len(libs_present) - 1:
            ax.legend(fontsize=7, title="method")
    fig.suptitle(title, y=1.02)
    fig.tight_layout(); plt.show()

for method_name in ("umap_leiden", "hierarchical"):
    df_m = cluster_maxdf[cluster_maxdf["Method"] == method_name]
    summ_m = cluster_summary[cluster_summary["Method"] == method_name]
    if df_m.empty:
        continue
    _hist_grid(df_m, summ_m, f"Per-cluster best-program enrichment — {method_name} "
                             f"(top-{CLUSTER_TOP_PER_CLUSTER}/cluster, ER>{ER_THRESH} = strong)")
    _kde_overlay(df_m, f"Per-cluster smoothed max-ER — {method_name}")


In [ ]:
# ============================================================
# Cache latent projections (+ denoised_gamma + W loadings from semantic_genept
# and semantic_geom) to AnnData for downstream (e.g. pyimpulse) analysis.
# Each model's latent goes into obsm["X_<model_name>"].
# denoised_gamma layer + model_dispersions in uns come from semantic_genept.
# denoised_gamma_geom + model_dispersions_geom come from semantic_geom.
# W_semantic_genept and W_semantic_geom in varm come from their respective models.
# ============================================================
import types

adata_proj = adata.copy()
for name, m in models.items():
    z = np.asarray(m.get_latent_representation())
    adata_proj.obsm[f"X_{name}"] = z
    print(f"{name}: obsm['X_{name}'] = {z.shape}")

# Patch get_normalized_expression: LDVAE.generative() uses **kwargs which hides
# transform_batch from inspect.signature() -> base RNASeqMixin raises
# NotImplementedError. Mirror pyimpulse's _patch_get_normalized_expression.
def _fixed_get_transform_batch_gen_kwargs(self, batch):
    return {"transform_batch": batch}

semantic_genept_model._get_transform_batch_gen_kwargs = types.MethodType(
    _fixed_get_transform_batch_gen_kwargs, semantic_genept_model
)
semantic_geneformer_model._get_transform_batch_gen_kwargs = types.MethodType(
    _fixed_get_transform_batch_gen_kwargs, semantic_geneformer_model
)

# Denoised expression + dispersions from SemanticSCVI (GenePT prior).
denoised = np.asarray(
    semantic_genept_model.get_normalized_expression(
        adata_sem_genept, library_size=10_000
    )
)
adata_proj.layers["denoised_gamma"] = denoised
print(f"layers['denoised_gamma'] = {denoised.shape} (from semantic_genept)")

try:
    px_r = semantic_genept_model.module.px_r.detach().exp().cpu().numpy()
    adata_proj.uns["model_dispersions"] = px_r
    print(f"uns['model_dispersions'] = {px_r.shape}")
except AttributeError:
    print("WARN: could not extract px_r dispersions from semantic_genept")

# Denoised expression + dispersions from SemanticSCVI (Geneformer prior) — geom.
denoised_geom = np.asarray(
    semantic_geneformer_model.get_normalized_expression(
        adata_sem_geneformer, library_size=10_000
    )
)
adata_proj.layers["denoised_gamma_geom"] = denoised_geom
print(f"layers['denoised_gamma_geom'] = {denoised_geom.shape} (from semantic_geom)")

try:
    px_r_geom = semantic_geneformer_model.module.px_r.detach().exp().cpu().numpy()
    adata_proj.uns["model_dispersions_geom"] = px_r_geom
    print(f"uns['model_dispersions_geom'] = {px_r_geom.shape}")
except AttributeError:
    print("WARN: could not extract px_r dispersions from semantic_geom")

# Gene loadings (W, shape n_genes x n_latent) from SemanticSCVI (GenePT prior).
W_genept = semantic_genept_model.get_loadings()  # DataFrame indexed by gene
W_genept = W_genept.reindex(adata_proj.var_names)
adata_proj.varm["W_semantic_genept"] = W_genept.values
adata_proj.uns["W_semantic_genept_columns"] = list(W_genept.columns)
print(f"varm['W_semantic_genept'] = {adata_proj.varm['W_semantic_genept'].shape}")

# Gene loadings from SemanticSCVI (Geneformer prior).
W_geom = semantic_geneformer_model.get_loadings()
W_geom = W_geom.reindex(adata_proj.var_names)
adata_proj.varm["W_semantic_geom"] = W_geom.values
adata_proj.uns["W_semantic_geom_columns"] = list(W_geom.columns)
print(f"varm['W_semantic_geom'] = {adata_proj.varm['W_semantic_geom'].shape}")

projections_path = NB_DIR / f"{ADATA_PATH.stem}_projections.h5ad"
adata_proj.write_h5ad(projections_path)
print(f"Projections written: {projections_path}")


In [ ]:
from datetime import datetime

notes = (
    f"SemanticSCVI [Geneformer]: {SEMANTIC_GENEFORMER_KWARGS['loss_mode']} loss, "
    f"cw={SEMANTIC_GENEFORMER_KWARGS['coherence_weight']}, "
    f"warmup={SEMANTIC_GENEFORMER_WARMUP_EPOCHS}/{SEMANTIC_GENEFORMER_MAX_EPOCHS} ep. "
    f"SemanticSCVI [GenePT-3-large]: {SEMANTIC_GENEPT_KWARGS['loss_mode']} loss, "
    f"cw={SEMANTIC_GENEPT_KWARGS['coherence_weight']}, "
    f"warmup={SEMANTIC_GENEPT_WARMUP_EPOCHS}/{SEMANTIC_GENEPT_MAX_EPOCHS} ep. "
    f"LDVAE+: weights_positive=True, {LDVAE_MAX_EPOCHS} ep. "
    f"scHPF: k={N_FACTORS}. cNMF: sklearn NMF k={N_FACTORS}, init={NMF_INIT}. "
    f"EXPIMAP: {len(EXPIMAP_MASK_TERMS)} HALLMARK terms (CD8-themed), alpha={EXPIMAP_ALPHA}, "
    f"alpha_kl={EXPIMAP_ALPHA_KL}, {EXPIMAP_N_EPOCHS} ep. "
    f"Benchmark: per_projection_n_top={PER_PROJECTION_N_TOP}, "
    f"cluster_n_top={CLUSTER_N_TOP}, cluster_top_per_cluster={CLUSTER_TOP_PER_CLUSTER}."
)
report_path = build_report(OUT_DIR, MODEL_NAMES, adata.shape, notes=notes)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
final_name = f"{ADATA_PATH.stem}_{ts}.html"
final_path = OUT_DIR / final_name
report_path.rename(final_path)

PRESERVE_DIRS = {"_llm_cache", "_score_cache"}
for child in OUT_DIR.iterdir():
    if child == final_path:
        continue
    if child.is_dir():
        if child.name in PRESERVE_DIRS:
            continue
        import shutil
        shutil.rmtree(child)
    else:
        if child.suffix == ".html":
            continue
        child.unlink()

print(f"Report written: {final_path}")
print(f"Size: {final_path.stat().st_size / 1e6:.1f} MB")